# Mesh 3 · Data on a mesh

**Fundamentals, part 3 of 4.** Companion to [Week 05 §5, §7, §9–10](../README.md).
Do [`02_walking_a_mesh`](02_walking_a_mesh.ipynb) first.

So far the mesh has been pure connectivity. A design model also needs to
say *how big*, *which way*, *what material*, *how loaded*. A mesh does this
in two ways: geometry it **computes** on demand, and attributes you
**store** on its vertices, faces and edges.

| You will | Section |
| --- | --- |
| measure every element | 1 |
| move a vertex, and see what does not change | 2 |
| attach your own data, with defaults | 3 |
| read it back in bulk, and query it | 4 |
| turn data into colour | 5 |
| put data on edges | 6 |
| save it, and find out which format keeps what | 7 |

In [ ]:
# Setup: run this cell first. It finds the shared drawing helper (viz.py),
# whether this notebook sits in its lecture folder or is your copy in MyWork/.
import sys
import pathlib

HERE = pathlib.Path.cwd()
NB = next(p / "Lecture" / "notebooks" for p in [HERE, *HERE.parents]
          if (p / "Lecture" / "notebooks" / "viz.py").exists())
sys.path.insert(0, str(NB))
from viz import chart, draw_graph, draw_mesh, draw_plan, draw_points, ramp, row, view3d

import math

from compas.datastructures import Mesh


def grid(nx, ny, spacing=1.0):
    """The grid from notebook 01: keys along x first, every face counter-clockwise."""
    mesh = Mesh()
    key = {}
    for j in range(ny + 1):
        for i in range(nx + 1):
            key[i, j] = mesh.add_vertex(x=i * spacing, y=j * spacing, z=0)
    for j in range(ny):
        for i in range(nx):
            mesh.add_face([key[i, j], key[i + 1, j], key[i + 1, j + 1], key[i, j + 1]])
    return mesh

## 1 · Geometry is computed, not stored

![Face centroids and normals on a wavy grid, and the same grid coloured by face area](../figures/mesh-measure.svg)

The mesh stores only vertex positions. Areas, normals, lengths and
centroids are **worked out when you ask**, from the positions as they are
at that moment. So they can never be out of date.

In [ ]:
roof = grid(6, 4, 0.5)
for k in roof.vertices():
    x, y, _ = roof.vertex_coordinates(k)
    roof.vertex_attribute(k, "z", 0.7 * math.sin(2 * math.pi * x / 3.0) + 0.15 * y)

f, e, v = 7, (8, 9), 9
print(f"face {f}:   area {roof.face_area(f):.3f} m2, centroid {[round(c, 2) for c in roof.face_centroid(f)]}")
print(f"           normal {[round(c, 2) for c in roof.face_normal(f)]}")
print(f"edge {e}: length {roof.edge_length(e):.3f} m, midpoint {[round(c, 2) for c in roof.edge_midpoint(e)]}")
print(f"vertex {v}: its share of the surface {roof.vertex_area(v):.3f} m2, normal {[round(c, 2) for c in roof.vertex_normal(v)]}")
print(f"whole mesh: {roof.area():.2f} m2 of surface over {6 * 0.5 * 4 * 0.5:.2f} m2 of plan")
draw_mesh(roof, "a wavy roof", view="iso", vertex_labels=False)

`vertex_area` deserves a second look: it is the part of the surface that
"belongs" to a vertex, a third of each triangle or a quarter of each quad
around it. Add them all up and you get the whole area. That is how a load
per square metre becomes a force on each node, which notebook 08 needs.

In [ ]:
print(f"sum of vertex areas {sum(roof.vertex_area(k) for k in roof.vertices()):.4f}  =  mesh area {roof.area():.4f}")

## 2 · Moving a vertex changes geometry, never topology

![The same grid before and after a sine-wave deformation; vertex keys 0 and 8 are in the same places in the structure](../figures/mesh-deform.svg)

`x`, `y`, `z` are attributes like any other. Set one, and every measure
from section 1 follows. No face is rebuilt, no neighbour changes.

In [ ]:
before = (roof.face_area(f), roof.vertex_neighbors(v), roof.euler())
roof.vertex_attribute(v, "z", roof.vertex_attribute(v, "z") + 0.5)
after = (roof.face_area(f), roof.vertex_neighbors(v), roof.euler())
print(f"area of face {f}:       {before[0]:.3f} -> {after[0]:.3f}   (changed)")
print(f"neighbours of vertex {v}: {before[1]} -> {after[1]}   (same)")
print(f"euler:                {before[2]} -> {after[2]}   (same)")
roof.vertex_attribute(v, "z", roof.vertex_attribute(v, "z") - 0.5)      # put it back

This split is why form finding, relaxation and simulation are all "move
the vertices in a loop": the structure is built once and only numbers
change (notebooks 07 and 10).

## 3 · Your own data, with defaults

![A deformed grid coloured by height, with the attribute dictionary of one vertex](../figures/mesh-attributes.svg)

Any name, any value, on any vertex, face or edge. Declare a **default**
first: then every element answers the question, including the ones you
never touched.

In [ ]:
roof.update_default_vertex_attributes(is_support=False, load=0.0)
roof.update_default_face_attributes(material="glass", thickness=0.012)

for k in roof.vertices():
    if roof.vertex_degree(k) == 2:                 # the four corners
        roof.vertex_attribute(k, "is_support", True)
roof.face_attribute(7, "material", "timber")
roof.face_attributes(8, ["material", "thickness"], ["timber", 0.040])

print("vertex 0: ", roof.vertex_attributes(0))
print("vertex 9: ", roof.vertex_attributes(9))
print("face 7:   ", roof.face_attributes(7))
print("face 0:   ", roof.face_attributes(0), " <- never set: the defaults")

Without the default, asking face 0 for its material would give `None`, and
the first `None` in a sum is an error a hundred lines later.

## 4 · Reading in bulk, and asking questions

One attribute from every element, in order, as a list:

In [ ]:
zs = roof.vertices_attribute("z")
print(f"{len(zs)} heights, from {min(zs):.2f} to {max(zs):.2f} m")
print("materials used:", sorted(set(roof.faces_attribute("material"))))

And the other direction: *which* elements have a given value. An exact
match, a `(low, high)` range, or any test you can write as a function.

In [ ]:
print("supports:           ", list(roof.vertices_where(is_support=True)))
print("timber panels:      ", list(roof.faces_where(material="timber")))
print("vertices above 0.8 m:", list(roof.vertices_where(z=(0.8, 99))))
steep = list(roof.faces_where_predicate(lambda face, attr: roof.face_normal(face)[2] < 0.75))
print("steep panels:       ", steep)

That last one mixes the two kinds of data: a **stored** key (the face) and
a **computed** value (its normal). Most useful queries do.

## 5 · From data to colour

A colour is one more attribute, or one more dictionary: element → colour.
Two classic maps, both one line of geometry each.

In [ ]:
lo, hi = min(zs), max(zs)
by_height = {face: ramp((roof.face_centroid(face)[2] - lo) / (hi - lo)) for face in roof.faces()}

sun = [0.5, -0.5, 0.707]                                       # from the south-east, 45 degrees up
by_sun = {face: ramp(max(0.0, sum(a * b for a, b in zip(roof.face_normal(face), sun)))) for face in roof.faces()}

row(draw_mesh(roof, "by height", view="iso", vertex_labels=False, face_colors=by_height, size=(440, 330)),
    draw_mesh(roof, "by how squarely it faces the sun", view="iso", vertex_labels=False, face_colors=by_sun, size=(440, 330)))

In [ ]:
from compas.colors import Color

view3d((roof, {"facecolor": {face: Color.from_hex(c) for face, c in by_sun.items()}, "show_edges": True}))

The second map is the start of every solar study: a normal, a sun vector,
a dot product. The mesh supplied the normal; you supplied one line.

## 6 · Edges carry data too

An edge is where two panels meet, so it is where joint data lives: a
profile, a length to cut, a force. Tag the edges of one loop as rails and
everything else as rungs (loops are from notebook 02).

In [ ]:
roof.update_default_edge_attributes(role="rung")
for start in [(0, 1), (14, 15), (28, 29)]:
    for edge in roof.edge_loop(start):
        roof.edge_attribute(edge, "role", "rail")

rails = list(roof.edges_where(role="rail"))
length = {role: sum(roof.edge_length(edge) for edge in roof.edges_where(role=role)) for role in ("rail", "rung")}
print(f"{len(rails)} rail edges, {length['rail']:.1f} m   |   rungs {length['rung']:.1f} m")
draw_mesh(roof, "rails (orange) and rungs", vertex_labels=False, highlight_edges=rails)

## 7 · Saving: which format keeps what

COMPAS's own JSON keeps **everything**: positions, faces, every attribute,
the defaults. Exchange formats such as OBJ keep the shape and nothing else.

In [ ]:
import tempfile

text = roof.to_jsonstring()
again = Mesh.from_jsonstring(text)
print(f"JSON: {len(text) / 1000:.0f} kB")
print("  supports survive:", list(again.vertices_where(is_support=True)))
print("  face 8 survives: ", again.face_attributes(8))

with tempfile.TemporaryDirectory() as folder:
    path = pathlib.Path(folder) / "roof.obj"
    roof.to_obj(str(path))
    plain = Mesh.from_obj(str(path))
print(f"OBJ:  V {plain.number_of_vertices()}, F {plain.number_of_faces()}: the shape came back")
print("  supports survive:", list(plain.vertices_where(is_support=True)))
print("  face 8's material:", plain.face_attribute(8, "material"))

So: **JSON between your own steps**, OBJ (or STL, PLY) to hand geometry
to a program that does not speak COMPAS. This is the "JSON artifact" in
the course's *core → artifact → adapter* pattern (Week 09): the mesh with
all its data is the thing you pass on. `mesh.to_json("roof.json")` writes
the file.

## What to keep

- Positions are stored. Areas, normals and lengths are computed, fresh,
  every time.
- Moving vertices never changes who is next to whom.
- Set a **default** before you set a value.
- `*_attribute` reads one, `*s_attribute` reads all, `*s_where` asks which.
- Data lives where it belongs: loads on vertices, materials on faces,
  joints on edges.

Next: [`04_changing_a_mesh`](04_changing_a_mesh.ipynb).

## Your turn
1. Store each face's area as an attribute, then list the five largest
   panels using `faces_attribute` and `sorted`.
2. Give every vertex a `load` equal to 1.5 kN/m² times its `vertex_area`.
   What should the loads sum to? Check.
3. Tag the faces of one `edge_strip` with `material="steel"` and total the
   area per material.
4. Change the sun vector to a low winter sun and redraw. Which panels
   change most?